In [3]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

nombre_carpeta = "deteccion_defectos-1"

# os.path.isdir verifica la existencia y que sea una carpeta al mismo tiempo
if os.path.isdir(nombre_carpeta):
    print(f"¡La carpeta '{nombre_carpeta}' existe!")
else:
    print(f"La carpeta '{nombre_carpeta}' no existe, se descargará de roboflow.")
    # Carga las variables del archivo .env
    load_dotenv()
    
    # Obtiene la API Key de las variables de entorno
    ROBOFLOW_KEY = os.getenv("API_ROBOFLOW")
    ROBOFLOW_WORKSPACE = os.getenv("API_ROBOFLOW_WORKSPACE")
    
    if not ROBOFLOW_KEY:
        raise ValueError("No se encontró la variable de entorno 'API_ROBOFLOW'. Asegúrate de definirla en tu archivo .env")
    rf = Roboflow(api_key=f"{ROBOFLOW_KEY}")
    project = rf.workspace("clothesdataset-yimbv").project("deteccion_defectos")
    version = project.version(1)
    dataset = version.download("yolov11")
    print(f"Dataset descargado de roboflow")

La carpeta 'deteccion_defectos-1' no existe, se descargará de roboflow.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to deteccion_defectos-1 in yolov11:: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 273/273 [00:00<00:00, 2288.78it/s]

Dataset descargado de roboflow


In [6]:
import yaml
import os

# Define las rutas absolutas o relativas desde donde ejecutarás el entrenamiento
# Es mejor usar rutas completas para evitar errores de "File Not Found"
dataset_path = os.path.abspath(f"{nombre_carpeta}")

data_config = {
    'path': dataset_path,      # Directorio raíz del dataset
    'train': 'train/images',   # Ruta relativa a 'path' para entrenamiento
    'val': 'valid/images',     # Ruta relativa a 'path' para validación
    'test': 'test/images',     # Ruta relativa a 'path' para pruebas (opcional)

    'nc': 2,                   # Número de clases
    'names': ['Corrido', 'Hueco'] # Asegúrate de que este orden sea el mismo de Roboflow
}

# Guardar el archivo
with open('data.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("Archivo data.yaml creado con éxito.")

Archivo data.yaml creado con éxito.


In [10]:
import torch
from ultralytics import YOLO
import os
import gc

def clear_gpu():
    torch.cuda.empty_cache()
    gc.collect()
    # Forzamos a que PyTorch no sea tan rígido con la memoria en la serie 50
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if __name__ == '__main__':
    clear_gpu()

    # Cargamos la versión 11 Medium
    model = YOLO('yolo11m.pt') 
    
    # Entrenamiento
    model.train(
        data="data.yaml",
        epochs=150,          # Al ser un dataset pequeño, dale tiempo a converger
        imgsz=1280,          # Tu resolución objetivo para conservar detalles pequeños
        batch=2,             # Bajamos a 2 para que quepa cómodamente en tus 8GB de VRAM
        amp=True,            # Fuerza precisión mixta (FP16) reduciendo el consumo a la mitad
        workers=2,           # Bajamos los workers para no saturar la transferencia CPU-GPU
        
        # --- Palancas de Mosaico y Aumentación ---
        mosaic=1.0,          # Activa al 100% la aumentación en mosaico de 4 imágenes
        mixup=0.15,          # Mezcla dos imágenes superpuestas (ayuda con texturas textiles)
        copy_paste=0.3,      # Clave: Pega defectos de una foto en otra para balancear 'hueco'
        
        # --- Palancas de Balanceo y Enfoque ---
        box=7.5,             # Le da prioridad milimétrica a la precisión de las cajas
        cls=1.5,             # Compensa el desbalance dándole 150% de peso a la clasificación
        
        # --- Variación de Color y Entorno ---
        hsv_h=0.15,          # Varía el tono para cubrir la diversidad de hilos/colores
        hsv_s=0.7,           # Varía la saturación
        hsv_v=0.4,           # Varía el brillo para emular sombras en el tejido

        # --- HARDWARE ---
        cache=False,
        device=0,
        name='defectos_v11m'
    )

New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.15, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=defectos_v11m, nbs=64, nms=False, opset=None,

In [13]:
metrics = model.val(conf=0.3)  # Filtra detecciones dudosas para maximizar precisión
print(metrics.results_dict['metrics/precision(B)'])

Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 1952.2552.0 MB/s, size: 550.6 KB)
val: Scanning C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\deteccion_defectos-1\valid\labels.cache... 20 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.5s/it 3.0s9.1s
                   all         20        381      0.824      0.546      0.693      0.323
               Corrido         13        295      0.803      0.651      0.749      0.381
                 Hueco         13         86      0.844      0.442      0.637      0.265
Speed: 7.6ms preprocess, 21.8ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\USER\Desktop\clothes-failures-detection\runs\detect\val4
0.8238958623895862


# Prueba

In [9]:
import cv2
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm

def tile_dataset(img_dir, label_dir, output_img_dir, output_label_dir, tile_size=1280, overlap=0.2):
    Path(output_img_dir).mkdir(parents=True, exist_ok=True)
    Path(output_label_dir).mkdir(parents=True, exist_ok=True)
    
    img_files = list(Path(img_dir).glob("*.jpg"))
    stride = int(tile_size * (1 - overlap)) # Desplazamiento considerando el solape

    for img_path in tqdm(img_files, desc="Procesando Tiling"):
        # 1. Cargar imagen original 4K
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w, _ = img.shape
        
        # 2. Cargar etiquetas originales (formato YOLO: class x_cen y_cen width height)
        label_path = Path(label_dir) / f"{img_path.stem}.txt"
        bboxes = []
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) == 5:
                        bboxes.append([int(parts[0])] + [float(x) for x in parts[1:]])

        # 3. Recorrer la imagen en una cuadrícula (Grid)
        tile_count = 0
        for y in range(0, h - tile_size + 1, stride):
            for x in range(0, w - tile_size + 1, stride):
                
                # Coordenadas absolutas en píxeles del parche actual
                x1, y1 = x, y
                x2, y2 = x + tile_size, y + tile_size
                
                # Extraer el parche físico de la imagen
                tile_img = img[y1:y2, x1:x2]
                tile_bboxes = []
                
                for bbox in bboxes:
                    cls, x_cen, y_cen, w_box, h_box = bbox
                    
                    # Convertir coordenadas normalizadas originales a píxeles absolutos de la imagen 4K
                    abs_x_cen = x_cen * w
                    abs_y_cen = y_cen * h
                    abs_w = w_box * w
                    abs_h = h_box * h
                    
                    # Calcular extremos (esquinas) de la caja del defecto original
                    box_x1 = abs_x_cen - (abs_w / 2)
                    box_y1 = abs_y_cen - (abs_h / 2)
                    box_x2 = abs_x_cen + (abs_w / 2)
                    box_y2 = abs_y_cen + (abs_h / 2)
                    
                    # --- NUEVA LÓGICA DE INTERSECCIÓN ---
                    # Calcular el área donde se cruzan el parche actual y el defecto
                    int_x1 = max(box_x1, x1)
                    int_y1 = max(box_y1, y1)
                    int_x2 = min(box_x2, x2)
                    int_y2 = min(box_y2, y2)
                    
                    int_w = max(0, int_x2 - int_x1)
                    int_h = max(0, int_y2 - int_y1)
                    
                    # Si hay un cruce real de píxeles entre el defecto y este parche
                    if int_w > 0 and int_h > 0:
                        area_interseccion = int_w * int_h
                        area_original = abs_w * abs_h
                        
                        # Filtro opcional: Evitamos guardar fragmentos insignificantes (menos del 5% visible)
                        # para no meter ruido visual o bboxes vacíos por error de un píxel en el borde.
                        if (area_interseccion / area_original) > 0.05:
                            
                            # Calcular coordenadas de la caja recortada RELATIVAS al origen del parche
                            new_x1 = int_x1 - x1
                            new_y1 = int_y1 - y1
                            new_x2 = int_x2 - x1
                            new_y2 = int_y2 - y1
                            
                            # Convertir a nuevo formato YOLO normalizado (0 a 1) relativo al tamaño del parche (1280)
                            new_x_cen = ((new_x1 + new_x2) / 2) / tile_size
                            new_y_cen = ((new_y1 + new_y2) / 2) / tile_size
                            new_w = (new_x2 - new_x1) / tile_size
                            new_h = (new_y2 - new_y1) / tile_size
                            
                            tile_bboxes.append([cls, new_x_cen, new_y_cen, new_w, new_h])

                # 4. Guardar el parche y su archivo de texto correspondiente
                tile_name = f"{img_path.stem}_tile_{tile_count}"
                cv2.imwrite(os.path.join(output_img_dir, f"{tile_name}.jpg"), tile_img)
                
                # Se escribe el archivo txt siempre (si está vacío, YOLO lo asimila perfectamente como fondo)
                with open(Path(output_label_dir) / f"{tile_name}.txt", 'w') as f:
                    for t_box in tile_bboxes:
                        f.write(f"{t_box[0]} {' '.join(map(str, t_box[1:]))}\n")
                        
                tile_count += 1

In [10]:
# Ejecutar Tiling para el set de ENTRENAMIENTO
tile_dataset(
    img_dir='deteccion_defectos-1/train/images',
    label_dir='deteccion_defectos-1/train/labels',
    output_img_dir='dataset_tiled/train/images',
    output_label_dir='dataset_tiled/train/labels',
    tile_size=1280,
    overlap=0.2 # 20% de solapamiento
)

# Ejecutar Tiling para el set de VALIDACIÓN
tile_dataset(
    img_dir='deteccion_defectos-1/valid/images',
    label_dir='deteccion_defectos-1/valid/labels',
    output_img_dir='dataset_tiled/valid/images',
    output_label_dir='dataset_tiled/valid/labels',
    tile_size=1280,
    overlap=0.2
)
# Ejecutar Tiling para el set de TEST
tile_dataset(
    img_dir='deteccion_defectos-1/test/images',
    label_dir='deteccion_defectos-1/test/labels',
    output_img_dir='dataset_tiled/test/images',
    output_label_dir='dataset_tiled/test/labels',
    tile_size=1280,
    overlap=0.2
)

Procesando Tiling: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  9.54it/s]


In [18]:
import yaml
import os

# Define las rutas absolutas o relativas desde donde ejecutarás el entrenamiento
# Es mejor usar rutas completas para evitar errores de "File Not Found"
dataset_path = os.path.abspath("dataset_tiled")

data_config = {
    'path': dataset_path,      # Directorio raíz del dataset
    'train': 'train/images',   # Ruta relativa a 'path' para entrenamiento
    'val': 'valid/images',     # Ruta relativa a 'path' para validación
    'test': 'test/images',     # Ruta relativa a 'path' para pruebas (opcional)

    'nc': 2,                   # Número de clases
    'names': ['Corrido', 'Hueco'] # Asegúrate de que este orden sea el mismo de Roboflow
}

# Guardar el archivo
with open('data_tiled.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("Archivo data_tiled.yaml creado con éxito.")

Archivo data_tiled.yaml creado con éxito.


In [11]:
import os
# 1. ESTO SIEMPRE PRIMERO: Configuración estricta de memoria para la serie RTX 50
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from ultralytics import YOLO
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

if __name__ == '__main__':
    clear_gpu()

    # Cargamos tu modelo entrenado con imágenes completas en 4K
    ruta_mejor_modelo = r"C:\Users\USER\Desktop\clothes-failures-detection\runs\detect\defectos_v11m\weights\best.pt"
    model = YOLO(ruta_mejor_modelo) 
    
    # Lanzamos el ajuste fino sobre el nuevo dataset con tiling por intersección
    model.train(
        data="data_tiled.yaml",    # Apunta a tus nuevos parches corregidos de 1280x1280
        epochs=120,                # Para un fine-tuning con 100-120 épocas suele ser más que suficiente
        imgsz=1280,                # Coincidencia 1:1 con tus parches para no perder resolución
        batch=2,                   # Seguro para tus 8GB de VRAM en imágenes de este tamaño
        amp=True,                  # Precisión mixta activa
        workers=2,
        
        # --- Configuración Quirúrgica de Fine-Tuning ---
        lr0=0.001,                 # Tasa de aprendizaje baja para conservar el conocimiento del modelo 4K
        lrf=0.01,                  # Cierre suave
        warmup_epochs=3,           # CRUCIAL: 3 épocas para que el optimizador se adapte al cambio de escala del tiling
        
        # --- Balance de Pérdidas (Loss Weights) ---
        box=7.5,                   # Precisión para el encuadre del defecto
        cls=1.5,                   # Penaliza errores de clase (ayuda a limpiar falsos positivos)
        
        # --- Aumentaciones Aptas para Textiles ---
        mosaic=1.0,                # Mantenlo activo; ayuda a procesar defectos que queden cortados en las esquinas
        mixup=0.0,                 # DESACTIVADO: Evita patrones extraños que confundan el tejido
        copy_paste=0.0,            # DESACTIVADO: Evita artefactos visuales falsos
        fliplr=0.5,                # Volteo horizontal
        flipud=0.5,                # Volteo vertical
        
        # --- HARDWARE y REPOSITORIO ---
        cache=False,
        device=0,
        name='defectos_v11m_tiled_v2_fijado', # Nombre de la carpeta para esta nueva etapa
        verbose=False              # Evita que Jupyter colapse por el canal IOPub
    )

New https://pypi.org/project/ultralytics/8.4.53 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_tiled.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\USER\Desktop\clothes-failures-detection\runs\detect\defectos_v11m\weights\best.pt, momentum=0

In [5]:
import os
# 1. ESTO SIEMPRE PRIMERO: Configuración estricta de memoria para la serie RTX 50
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from ultralytics import YOLO
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

clear_gpu()

ruta_mejor_modelo = r"C:\Users\USER\Desktop\clothes-failures-detection\runs\detect\defectos_v11m_tiled_90plus\weights\best.pt"
model = YOLO(ruta_mejor_modelo)

metrics = model.val(conf=0.3)  # Filtra detecciones dudosas para maximizar precisión
print(metrics.results_dict['metrics/precision(B)'])
print(metrics.results_dict['metrics/recall(B)'])

Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
YOLO11m summary (fused): 125 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1959.4535.9 MB/s, size: 276.4 KB)
val: Scanning C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\dataset_tiled\valid\labels.cache... 60 images, 22 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.4s/it 9.5s3.9sss
                   all         60        399      0.828      0.699      0.801      0.399
               Corrido         34        309      0.849      0.654       0.78      0.429
                 Hueco         28         90      0.807      0.744      0.822      0.368
Speed: 7.8ms preprocess, 123.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to C:\Users\USER\Desktop\clothes-failures-detection\runs\detect